# report of energy comparing for ml and ref Nanoscale, 2017, 9, 7143 SI
and save all 5a, 5b to energies_ml.csv

In [6]:
import pandas as pd
from pathlib import Path
from lzn.universal import *

# from pandas.conftest import axis


### reference energies

In [7]:
energies_path = to_path("~/TiO2/tio2_ChenDixon_xyz/energies.csv")
df_ref = pd.read_csv(energies_path)
df_ref = df_ref[["id", "energy_eV"]]

In [8]:
ref_e5 = df_ref["energy_eV"][0]
ref_e8 = df_ref["energy_eV"][2]
ref_e40 = df_ref["energy_eV"][4]
ref_e64 = df_ref["energy_eV"][7]


df_ref.loc[0:1, "energy_eV"] -= ref_e5
df_ref.loc[2:3, "energy_eV"] -= ref_e8
df_ref.loc[4:6, "energy_eV"] -= ref_e40
df_ref.loc[7:8, "energy_eV"] -= ref_e64


In [9]:
df_ref

,id,energy_eV
0,1a,0.000000e+00
1,5a,-1.088537e+05
2,5b,0.000000e+00
3,8a,-8.164206e+04
4,8b,0.000000e+00
5,40a,-8.708478e+05
6,40b,-8.708476e+05
7,40c,0.000000e+00
8,64a,-6.531385e+05
9,64b,-1.741687e+06


### ml energies

In [10]:
mlE_dir = to_path("~/TiO2/tio2_ChenDixon_out")
modals = [x for x in mlE_dir.iterdir() if x.is_dir()]
mlE_path = [x / "energies.csv" for x in modals]

print(modals)
print(mlE_path)

df_ml = [pd.read_csv(x) for x in mlE_path]

# file values like: "40a.xyz", "5b.xyz", "64b.xyz"
df_ml_sorted = []
for df in df_ml:
    tmp = df["file"].str.extract(r"^(\d+)([A-Za-z])", expand=True)  # number, letter
    df_sorted = (
        df.assign(_n=tmp[0].astype(int), _s=tmp[1].str.lower())
          .sort_values(["_n", "_s"])
          .drop(columns=["_n", "_s"])
          .reset_index(drop=True)
    )
    df_ml_sorted.append(df_sorted)


[PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/matpes_pbe'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/omat24'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/omol25_low'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/mpa'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/matpes_r2scan')]
[PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/matpes_pbe/energies.csv'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/omat24/energies.csv'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/omol25_low/energies.csv'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/mpa/energies.csv'), PosixPath('/home/zining/TiO2/tio2_ChenDixon_out/matpes_r2scan/energies.csv')]


In [11]:
df_ml_sorted[3]

,file,E_initial,E_final,dE
0,1a.xyz,-21.115004,-21.125584,-0.010580
1,5a.xyz,-125.446930,-125.493195,-0.046265
2,5b.xyz,-125.427109,-125.462975,-0.035866
3,8a.xyz,-204.973022,-205.026489,-0.053467
4,8b.xyz,-204.893677,-204.944855,-0.051178
5,40a.xyz,-1053.468384,-1053.583252,-0.114868
6,40b.xyz,-1052.743408,-1052.850586,-0.107178
7,40c.xyz,-1049.600586,-1050.026123,-0.425537
8,64a.xyz,-1687.843262,-1687.996094,-0.152832
9,64b.xyz,-1688.939209,-1689.117798,-0.178589


###### Save ml energies to energies_ml.csv

In [ ]:
save_path = Path("energies_ml.csv")
df_save = pd.read_csv(save_path)

for i, modal_i_path in enumerate(mlE_path):
    modal_i = modal_i_path.parent.stem
    df_ml_i = df_ml_sorted[i]
    ini_5a, ini_5b = df_ml_i.loc[0, 'E_initial'], df_ml_i.loc[1, 'E_initial']
    fin_5a, fin_5b = df_ml_i.loc[0, 'E_final'], df_ml_i.loc[1, 'E_final']
    df_save.loc[2*i] = [modal_i, int(0), ini_5a, ini_5b]
    df_save.loc[2*i+1] = [modal_i, int(1), fin_5a, fin_5b]

df_save
df_save.to_csv(save_path, index=False)


###### initial energies by ml ff

In [ ]:
for x in df_ml_sorted:
    ref_e5 = x.loc[0, "E_initial"]
    ref_e8 = x.loc[2, "E_initial"]
    ref_e40 = x.loc[4, "E_initial"]
    ref_e64 = x.loc[7, "E_initial"]

    print(ref_e5)

    x.loc[0:1, "E_initial"] -= ref_e5
    x.loc[2:3, "E_initial"] -= ref_e8
    x.loc[4:6, "E_initial"] -= ref_e40
    x.loc[7:8, "E_initial"] -= ref_e64

In [ ]:
df_ml_norm = [x["E_initial"] for x in df_ml_sorted]
df_ml_norm[1]

In [ ]:
col_names = [str(x.name) for x in modals]
print(col_names)
df_all = pd.concat(df_ml_norm, axis=1)
df_all.columns = col_names

In [ ]:
df_allx = pd.concat([df_ref, df_all], axis=1)
df_allx = df_allx.rename(columns={"energy_eV": "ref_eV"})

In [ ]:
df_allx

In [ ]:
import matplotlib.pyplot as plt

num = 9 # plot first num structures
labels = [5, 8, 40, 64]

plt.clf()
x = range(num)
ycols = df_allx.columns[2:]             # number of columns (modals)

fig, ax = plt.subplots()

for col in ycols:
    plt.plot(x, df_allx[col][:num].to_numpy(), 'o', label=col)
plt.plot(x, df_allx["ref_eV"][:num].to_numpy(), 'rx', label="ref_eV", alpha=1)

plt.xlabel("Index")
plt.ylabel("E/ eV")
plt.grid(True, linestyle='--', alpha=0.3)

xs = [0, 2, 4, 7]
ax.set_xticks(xs)                              # put ticks at these x values
ax.grid(True, axis="x", which="major", linestyle="-", alpha=0.5)  # solid vertical grid lines
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0))  # outside right
ax.set_xticklabels(labels)

fig.tight_layout()
plt.show()


###### final ml energy after ml opt
5a and 5b

In [ ]:
df_ml_final_norm = []
for x in df_ml_sorted:
    ref_e5 = x["E_final"][0]
    df_ml_final_norm.append(x.loc[0:1, "E_final"] - ref_e5)

print(df_ml_final_norm)
df_ml_final_norm[1]

In [ ]:
col_names = [str(x.name) for x in modals]
print(col_names)
df_final_all = pd.concat(df_ml_final_norm, axis=1)
df_final_all.columns = col_names

df_final_allx = pd.concat([df_ref[0:2], df_final_all], axis=1)
df_final_allx = df_final_allx.rename(columns={"energy_eV": "ref_eV"})
df_final_allx

### dE optimization

In [ ]:
l_dE = []
for x in df_ml_sorted:
    l_dE.append(x["dE"])

df_dE = pd.concat(l_dE, axis=1)
df_dE.columns = col_names
df_dE = pd.concat([df_ref, df_dE], axis=1)

df_dE

In [ ]:
df_5 = df_dE.loc[0:1, :]
df_5

In [ ]:
df_5.loc[2, "id"] = "dE"
df_5.iloc[2, 1:] = df_5.iloc[1, 1:] - df_5.iloc[0, 1:]
df_5